# v2 validation check — KMeans → profitable clusters → LR (polynomial), draw market

Checks the candidate pipeline against `06_docs/decisions.md` (validation protocol v2), and compares **two ways
of drawing the cluster boundaries**:

- **running** — fit the boundaries on the point-in-time (running) season averages. Reflects the state we
  actually *see* at decision time; noisier early in the season.
- **final** — fit the boundaries on end-of-season stats (`team_season_final`), the stable "true character" of
  each team, then still assign each match by its point-in-time stats. Same idea as the cat3q thresholds
  (boundaries from the full-season distribution, value placed against them point-in-time).

Only the *boundaries* differ. Every match is always **assigned** and the bet model is always **fit** on
point-in-time (running) features, so both variants are honest at bet time.

Roles: **GATE** (hard pass/fail), **DIRECTIONAL** (can only veto), **DIAGNOSTIC** (information). Rule #0: which
clusters to keep and the model are fit on the training fold only.

## Decision: use the `running` map

Both boundary maps below pass every gate, so the signal does not depend on how the boundaries are drawn
(map-invariant) — reassuring in itself. **`running` is the deployed choice**, for three reasons:

- its **league x season breakdown is less extreme** than `final`'s. `final` leans harder on a few wild
  cells (e.g. spain 2324 +1.58 vs running +0.88); a lower-variance edge is the more trustworthy one.
- it is **stronger on walk-forward**, the time-honest, deployment-realistic check.
- it is the state **actually observable at bet time**. The mispricing we exploit is defined against what the
  market sees — the running form — not the hidden end-of-season character.

`final` is kept below only as a robustness comparison, not the deployed choice.

In [1]:
GROUP = "minor"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

CONFIG = {
    "n_clusters": 5,        # reference cluster count for the gates; k is swept separately below
    "buffer": 0.02,         # only bet when the model probability beats the implied one by this margin
    "min_game_number": 8,   # skip early-season matches, where the season stats are still too noisy
}

OUTCOME = "t_draw_flg"      # 1 when the match ended in a draw
DRAW_IMPLIED = {"mrkt": "mrkt_draw_impl", "b365": "b365_draw_impl"}  # implied draw prob = 1 / draw odds

# The four team-strength quantities, measured two ways.
# RUNNING: each team's average so far this season, as known before the match (point-in-time).
RUNNING_FEATURES = [
    "homet_season_goals_for_avg", "homet_season_goals_agst_avg",
    "awayt_season_goals_for_avg", "awayt_season_goals_agst_avg",
]
# FINAL: the same four quantities but taken from the finished season (team_season_final). Stable, but only
# usable to place the boundaries in hindsight, never to locate an individual match (that would be leakage).
FINAL_FEATURES = ["home_for_final", "home_agst_final", "away_for_final", "away_agst_final"]

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

# Attach each team's end-of-season goals for/against to every match it played, once as the home side and
# once as the away side. These columns feed only the "final" boundaries.
final_stats = pd.read_parquet(f"C:/Users/twluc/frame/01_data/02_features/{GROUP}/team_season_final.parquet")
final_stats = final_stats[["team", "season", "season_goals_for_avg", "season_goals_agst_avg"]]
home_final = final_stats.rename(columns={"team": "home_team",
    "season_goals_for_avg": "home_for_final", "season_goals_agst_avg": "home_agst_final"})
away_final = final_stats.rename(columns={"team": "away_team",
    "season_goals_for_avg": "away_for_final", "season_goals_agst_avg": "away_agst_final"})
abt = abt.merge(home_final, on=["home_team", "season"], how="left").merge(away_final, on=["away_team", "season"], how="left")

# Keep matches where both teams have enough history, and drop rows missing either the running OR the final
# features so both variants are compared on exactly the same set of matches.
matches = abt[
    (abt["homet_game_number"] > CONFIG["min_game_number"])
    & (abt["awayt_game_number"] > CONFIG["min_game_number"])
].dropna(subset=RUNNING_FEATURES + FINAL_FEATURES + list(DRAW_IMPLIED.values()) + [OUTCOME, "league", "season"]).copy()

CENTROID_MAPS = {"running": RUNNING_FEATURES, "final": FINAL_FEATURES}
print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 6507   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england2', 'france2', 'germany2', 'italy2', 'spain2']


In [2]:
# --- Scoring -----------------------------------------------------------------
# ROI under implied-probability staking: stake the implied probability on each bet and collect 1.0 on a win.
# ROI = (won - staked) / staked measures whether an edge exists independent of bet size.
def pooled_roi(implied, won):
    staked = implied.sum()
    return (won.sum() - staked) / staked if staked > 0 else np.nan

# ROI within each group, from summed stakes and wins (agg avoids groupby.apply, which varies across pandas).
def roi_grouped(bets, keys):
    totals = bets.groupby(keys).agg(won=("won", "sum"), staked=("implied", "sum"))
    return (totals["won"] - totals["staked"]) / totals["staked"]

# --- Cross-validation folds --------------------------------------------------
def season_folds(data):  # leave-one-season-out: each season is the test set once
    for season in sorted(data["season"].unique()):
        yield ~(data["season"] == season), data["season"] == season, season

def expanding_folds(data, min_train_seasons=2):  # walk-forward: train on earlier seasons only
    seasons = sorted(data["season"].unique())
    for split in range(min_train_seasons, len(seasons)):
        yield data["season"].isin(seasons[:split]), data["season"] == seasons[split], seasons[split]

# --- Pipeline pieces ---------------------------------------------------------
# Bet-signal model: expand the four running features into squares and pairwise products, standardise, fit.
def build_draw_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler(),
                         LogisticRegression(max_iter=3000))

# Given each training match's cluster label, keep the clusters where draws happened more often than the
# market priced them. Computed on training matches only, so the held-out fold never influences the choice.
def profitable_clusters(train, cluster_labels, min_matches=100):
    kept = []
    for cluster in np.unique(cluster_labels):
        in_cluster = cluster_labels == cluster
        if in_cluster.sum() < min_matches:
            continue
        in_cluster_rows = train[in_cluster]
        draw_edge = in_cluster_rows[OUTCOME].mean() - in_cluster_rows["mrkt_draw_impl"].mean()
        if draw_edge > 0:
            kept.append(cluster)
    return kept

In [3]:
# Run the pipeline honestly across a set of folds and return every bet placed on the held-out folds.
# `centroid_map` chooses what the cluster boundaries are fit on ("running" or "final"); matches are always
# ASSIGNED, and the model is always FIT, on running (point-in-time) features.
def collect_bets(folds, n_clusters, odds_source, centroid_map):
    boundary_features = CENTROID_MAPS[centroid_map]
    implied_column = DRAW_IMPLIED[odds_source]
    placed = []
    for train_mask, test_mask, fold_name in folds:
        train, test = matches[train_mask], matches[test_mask]

        # Boundaries from the chosen map, training seasons only. Both maps live in the same goals-per-match
        # units, so a match located by its running stats lands sensibly among final-based centroids too.
        kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(train[boundary_features].to_numpy())
        train_clusters = kmeans.predict(train[RUNNING_FEATURES].to_numpy())
        test_clusters = kmeans.predict(test[RUNNING_FEATURES].to_numpy())

        kept = profitable_clusters(train, train_clusters)
        in_region_test = np.isin(test_clusters, kept)
        if not in_region_test.any():
            continue
        in_region_train = np.isin(train_clusters, kept)

        model = build_draw_model()
        model.fit(train[in_region_train][RUNNING_FEATURES], train[in_region_train][OUTCOME].astype(int))

        region = test[in_region_test].copy()
        region["model_prob"] = model.predict_proba(region[RUNNING_FEATURES])[:, list(model.classes_).index(1)]
        region["implied"] = region[implied_column]
        region["won"] = region[OUTCOME].astype(int)
        region["fold"] = fold_name

        is_bet = region["model_prob"] > region["implied"] + CONFIG["buffer"]
        placed.append(region.loc[is_bet, ["implied", "won", "league", "season", "fold"]])
    return pd.concat(placed, ignore_index=True)

## Gate 1 — time robustness (leave-one-season-out)
Edge positive in at least 3 of 4 seasons. Both boundary maps, b365 odds.

In [4]:
for centroid_map in ["running", "final"]:
    bets = collect_bets(season_folds(matches), CONFIG["n_clusters"], "b365", centroid_map)
    per_season = roi_grouped(bets, "fold")
    positive = int((per_season > 0).sum())
    verdict = "PASS" if positive >= 3 else "FAIL"
    print(f"[{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          f"seasons +{positive}/{len(per_season)} {verdict}   ({len(bets)} bets)   "
          + "  ".join(f"{season}:{value:+.3f}" for season, value in per_season.items()))

[running] pooled -0.0526   seasons +1/3 FAIL   (431 bets)   2223:-0.156  2425:+0.034  2526:-0.055


[final  ] pooled -0.0567   seasons +1/3 FAIL   (415 bets)   2223:+0.099  2324:-0.084  2526:-0.088


## Gate 2 — league robustness (survive removing the best league)
Remove each league in turn; the best league is the one whose removal hurts most. Edge must stay positive
without it. Both boundary maps, b365 odds.

In [5]:
for centroid_map in ["running", "final"]:
    bets = collect_bets(season_folds(matches), CONFIG["n_clusters"], "b365", centroid_map)
    without_league = {league: pooled_roi(bets[bets["league"] != league]["implied"], bets[bets["league"] != league]["won"])
                      for league in bets["league"].unique()}
    best_league = min(without_league, key=without_league.get)
    survives = without_league[best_league] > 0
    per_league = roi_grouped(bets, "league").sort_values()
    print(f"[{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          f"best league = {best_league} -> without it {without_league[best_league]:+.4f}   "
          f"GATE 2 {'PASS' if survives else 'FAIL'}")
    print("          per league: " + "   ".join(f"{league}:{value:+.3f}" for league, value in per_league.items()))

[running] pooled -0.0526   best league = england2 -> without it -0.0851   GATE 2 FAIL
          per league: italy2:-0.220   spain2:-0.218   france2:+0.011   england2:+0.020   germany2:+0.030


[final  ] pooled -0.0567   best league = germany2 -> without it -0.0907   GATE 2 FAIL
          per league: italy2:-0.227   spain2:-0.218   france2:-0.041   england2:-0.021   germany2:+0.068


## Directional — walk-forward (time-honest) and knob robustness (sweep k)

In [6]:
print("walk-forward (b365):")
for centroid_map in ["running", "final"]:
    bets = collect_bets(expanding_folds(matches), CONFIG["n_clusters"], "b365", centroid_map)
    per_fold = roi_grouped(bets, "fold")
    vetoes = bool((per_fold < -0.03).any())
    print(f"  [{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          + "   ".join(f"{fold}:{value:+.3f}" for fold, value in per_fold.items())
          + f"   {'VETO' if vetoes else 'no veto'}")

print("\nknob robustness — pooled ROI (b365) by cluster count:")
print("   k  running   final")
for n_clusters in [4, 5, 6, 8, 10]:
    run_bets = collect_bets(season_folds(matches), n_clusters, "b365", "running")
    fin_bets = collect_bets(season_folds(matches), n_clusters, "b365", "final")
    print(f"  {n_clusters:2d}  {pooled_roi(run_bets['implied'], run_bets['won']):+.4f}  "
          f"{pooled_roi(fin_bets['implied'], fin_bets['won']):+.4f}")

walk-forward (b365):


  [running] pooled -0.0474   2425:-0.040   2526:-0.055   VETO


  [final  ] pooled -0.0591   2425:-0.032   2526:-0.088   VETO

knob robustness — pooled ROI (b365) by cluster count:
   k  running   final


   4  -0.0178  -0.0645


   5  -0.0526  -0.0567


   6  -0.0334  -0.0569


   8  +0.0016  -0.0688


  10  -0.0285  -0.0673


## Diagnostic — where the edge lives (league × season), per map
Not a gate. Shows how concentrated / noisy the edge is at the finest granularity, for each boundary map.

In [7]:
for centroid_map in ["running", "final"]:
    bets = collect_bets(season_folds(matches), CONFIG["n_clusters"], "b365", centroid_map)
    breakdown = roi_grouped(bets, ["league", "season"]).unstack("season").round(3)
    print(f"[{centroid_map}] ROI by league x season (b365), total {len(bets)} bets:")
    print(breakdown.to_string())
    print()

[running] ROI by league x season (b365), total 431 bets:
season     2223   2425   2526
league                       
england2  0.299 -0.038 -0.096
france2  -0.112  0.357 -0.097
germany2 -1.000  0.045  0.243
italy2   -0.356 -0.533  0.072
spain2   -0.568  0.048 -0.227



[final] ROI by league x season (b365), total 415 bets:
season     2223   2324   2526
league                       
england2  0.444 -0.178 -0.094
france2  -0.034 -0.010 -0.066
germany2 -1.000  0.134  0.188
italy2   -0.029 -0.137 -0.599
spain2    0.706 -0.455 -0.223



## Summary

In [8]:
# ---------------------------------------------------------------------------
# SUMMARY — gates, veto, and diagnostic spread at a glance (b365, running map, default k).
# ---------------------------------------------------------------------------
k = CONFIG.get("n_clusters", CONFIG.get("n_team_types"))
bets = collect_bets(season_folds(matches), k, "b365", "running")
walk = collect_bets(expanding_folds(matches), k, "b365", "running")

if bets.empty:
    print("SUMMARY: no bets placed under this pipeline")
else:
    per_season = roi_grouped(bets, "fold")
    gate1_pass = int((per_season > 0).sum()) >= 3
    without_league = {league: pooled_roi(bets[bets["league"] != league]["implied"], bets[bets["league"] != league]["won"])
                      for league in bets["league"].unique()}
    gate2_pass = min(without_league.values()) > 0
    wf_veto = (not walk.empty) and bool((roi_grouped(walk, "fold") < -0.03).any())
    spread = roi_grouped(bets, ["league", "season"]).std()  # how noisy the per-cell breakdown is
    print(f"SUMMARY (b365, running map, k={k})")
    print(f"  pooled ROI                     {pooled_roi(bets['implied'], bets['won']):+.4f}  ({len(bets)} bets)")
    print(f"  GATE 1  (LOSO >= 3/4 seasons)    {'PASS' if gate1_pass else 'FAIL'}  (+{int((per_season > 0).sum())}/{len(per_season)})")
    print(f"  GATE 2  (drop best league)       {'PASS' if gate2_pass else 'FAIL'}  (worst {min(without_league.values()):+.4f})")
    print(f"  walk-forward                     {'VETO' if wf_veto else 'no veto'}")
    print(f"  diagnostic spread (std of league x season ROI)   {spread:.3f}")

SUMMARY (b365, running map, k=5)
  pooled ROI                     -0.0526  (431 bets)
  GATE 1  (LOSO >= 3/4 seasons)    FAIL  (+1/3)
  GATE 2  (drop best league)       FAIL  (worst -0.0851)
  walk-forward                     VETO
  diagnostic spread (std of league x season ROI)   0.363
